# BM25 baseline locale

Questo notebook implementa una baseline BM25 sugli stessi dati usati per la semantic search, per confrontare i due approcci sulle stesse metriche.

Input:
- `data/processed/embedding_corpus.jsonl`
- `data/processed/embedding_queries.jsonl`

Output:
- `data/processed/bm25_results.jsonl`
- `data/processed/bm25_metrics.json`

Usa gli stessi file di input della semantic search baseline: il campo `embedding_text` contiene gia' la rappresentazione testuale pulita, che e' la stessa da dare a BM25.

Non vengono usati vector database, Weaviate o FastAPI.

## 1. Configurazione

In [ ]:
import json
import os
from pathlib import Path
from typing import Any

import numpy as np


current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
DATA_DIR = PROJECT_ROOT / "data" / "processed"


def configured_path(env_var: str, default_path: Path) -> Path:
    value = os.getenv(env_var)
    path = Path(value) if value else default_path
    return path if path.is_absolute() else PROJECT_ROOT / path


CORPUS_INPUT_PATH = configured_path("BM25_CORPUS_PATH", DATA_DIR / "embedding_corpus.jsonl")
QUERY_INPUT_PATH = configured_path("BM25_QUERY_PATH", DATA_DIR / "embedding_queries.jsonl")
RESULTS_OUTPUT_PATH = configured_path("BM25_RESULTS_PATH", DATA_DIR / "bm25_results.jsonl")
METRICS_OUTPUT_PATH = configured_path("BM25_METRICS_PATH", DATA_DIR / "bm25_metrics.json")

TOP_K = int(os.getenv("BM25_TOP_K", "5"))
if TOP_K <= 0:
    raise ValueError("BM25_TOP_K deve essere maggiore di zero")

EVALUATION_KS = [1, 3, 5]

CORPUS_INPUT_PATH, QUERY_INPUT_PATH, RESULTS_OUTPUT_PATH, METRICS_OUTPUT_PATH, TOP_K

## 2. Funzioni di utilita'

In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"File non trovato: {path}")
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON non valido in {path}, riga {line_number}") from exc
    return records


def write_jsonl_atomic(records: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    tmp_path.replace(path)


def write_json_atomic(data: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
        f.write("\n")
    tmp_path.replace(path)


def tokenize(text: str) -> list[str]:
    """Tokenizzazione semplice: lowercase e split su spazi.
    Scelta consapevole: niente stemming ne' rimozione stopword,
    per mantenere il confronto con gli embeddings il piu' diretto possibile.
    """
    return text.lower().split()

## 3. Installazione e import di rank-bm25

Se il pacchetto non e' gia' installato, eseguire:

```
pip install rank-bm25
```

In [ ]:
try:
    from rank_bm25 import BM25Okapi
except ImportError as exc:
    raise ImportError(
        "Installa rank-bm25 prima di eseguire questo notebook: "
        "pip install rank-bm25"
    ) from exc

print("rank-bm25 importato correttamente")

## 4. Lettura dati

In [ ]:
corpus_records = read_jsonl(CORPUS_INPUT_PATH)
query_records = read_jsonl(QUERY_INPUT_PATH)

print(f"Corpus records: {len(corpus_records):,}")
print(f"Query records:  {len(query_records):,}")

## 5. Costruzione indice BM25

`BM25Okapi` e' la variante piu' usata di BM25 ed e' la referenza standard nei confronti con approcci neurali.

In [ ]:
corpus_texts = [record["embedding_text"] for record in corpus_records]
corpus_tokenized = [tokenize(text) for text in corpus_texts]

bm25 = BM25Okapi(corpus_tokenized)

print(f"Indice BM25 costruito su {len(corpus_tokenized):,} documenti")

## 6. Ranking e costruzione risultati

Stesso schema di output di `semantic_search_baseline.ipynb`: il notebook di analisi errori funziona identico sui risultati BM25.

In [ ]:
corpus_index_by_record_id = {
    record["record_id"]: index
    for index, record in enumerate(corpus_records)
}

results = []

for query_record in query_records:
    query_text = query_record["embedding_text"]
    query_tokens = tokenize(query_text)

    scores = bm25.get_scores(query_tokens)
    sorted_indices = np.argsort(-scores)
    candidate_indices = sorted_indices[:TOP_K]

    correct_corpus_index = corpus_index_by_record_id.get(query_record["record_id"])
    if correct_corpus_index is None:
        correct_rank = None
        reciprocal_rank = 0.0
    else:
        correct_score = scores[correct_corpus_index]
        correct_rank = int(1 + np.sum(scores > correct_score))
        reciprocal_rank = 1.0 / correct_rank

    candidates = [
        {
            "record_id": corpus_records[int(idx)]["record_id"],
            "source_image": corpus_records[int(idx)].get("source_image"),
            "bm25_score": float(scores[idx]),
        }
        for idx in candidate_indices
    ]

    metadata = query_record.get("metadata", {})
    results.append({
        "record_id": query_record["record_id"],
        "source_image": query_record.get("source_image"),
        "metadata": metadata,
        "evaluation_split": metadata.get("evaluation_split") if isinstance(metadata, dict) else None,
        "correct_rank": correct_rank,
        "reciprocal_rank": reciprocal_rank,
        "is_match_top_1": correct_rank is not None and correct_rank <= 1,
        "is_match_top_3": correct_rank is not None and correct_rank <= 3,
        "is_match_top_5": correct_rank is not None and correct_rank <= 5,
        "candidates": candidates,
    })

print(f"Risultati creati: {len(results):,}")

## 7. Metriche

Stesse metriche della semantic search baseline: top-1/3/5 accuracy e MRR, sia complessive sia per `evaluation_split` e per `noise_level`.

In [ ]:
def compute_metrics(rows: list[dict[str, Any]]) -> dict[str, Any]:
    if not rows:
        return {"query_count": 0, "top_1_accuracy": None, "top_3_accuracy": None,
                "top_5_accuracy": None, "mrr": None}
    return {
        "query_count": len(rows),
        "top_1_accuracy": float(np.mean([r["is_match_top_1"] for r in rows])),
        "top_3_accuracy": float(np.mean([r["is_match_top_3"] for r in rows])),
        "top_5_accuracy": float(np.mean([r["is_match_top_5"] for r in rows])),
        "mrr": float(np.mean([r["reciprocal_rank"] for r in rows])),
    }


split_names = sorted({
    r.get("evaluation_split")
    for r in results
    if r.get("evaluation_split") is not None
})

noise_levels = ["low", "medium", "high"]

metrics = {
    "config": {
        "top_k": TOP_K,
        "tokenization": "lowercase + split",
        "bm25_variant": "BM25Okapi",
        "corpus_input_path": str(CORPUS_INPUT_PATH),
        "query_input_path": str(QUERY_INPUT_PATH),
    },
    "overall": compute_metrics(results),
    "by_evaluation_split": {
        split: compute_metrics([r for r in results if r.get("evaluation_split") == split])
        for split in split_names
    },
    "by_noise_level": {
        level: compute_metrics([
            r for r in results
            if isinstance(r.get("metadata"), dict)
            and r["metadata"].get("noise_level") == level
        ])
        for level in noise_levels
    },
}

metrics

## 8. Confronto diretto con la semantic search baseline

Tabella comparativa: BM25 vs embeddings sulle stesse metriche e gli stessi split.

In [ ]:
import pandas as pd


SEMANTIC_METRICS_PATH = DATA_DIR / "semantic_search_metrics.json"

if SEMANTIC_METRICS_PATH.exists():
    with SEMANTIC_METRICS_PATH.open("r", encoding="utf-8") as f:
        semantic_metrics = json.load(f)

    def metrics_row(label: str, m: dict[str, Any]) -> dict[str, Any]:
        return {
            "modello": label,
            "query": m.get("query_count"),
            "top_1_acc": round(m.get("top_1_accuracy") or 0, 4),
            "top_3_acc": round(m.get("top_3_accuracy") or 0, 4),
            "top_5_acc": round(m.get("top_5_accuracy") or 0, 4),
            "mrr": round(m.get("mrr") or 0, 4),
        }

    rows = [
        metrics_row("BM25 — totale", metrics["overall"]),
        metrics_row("Embeddings — totale", semantic_metrics["overall"]),
    ]
    for split in split_names:
        rows.append(metrics_row(f"BM25 — {split}", metrics["by_evaluation_split"].get(split, {})))
        rows.append(metrics_row(f"Embeddings — {split}", semantic_metrics.get("by_evaluation_split", {}).get(split, {})))
    for level in noise_levels:
        bm25_level = metrics["by_noise_level"].get(level, {})
        sem_level = semantic_metrics.get("by_noise_level", {}).get(level, {})
        if bm25_level.get("query_count"):
            rows.append(metrics_row(f"BM25 — noise:{level}", bm25_level))
        if sem_level.get("query_count"):
            rows.append(metrics_row(f"Embeddings — noise:{level}", sem_level))

    comparison_df = pd.DataFrame(rows)
    pd.set_option("display.max_colwidth", 30)
    display(comparison_df)
else:
    print(f"File semantic_search_metrics.json non trovato in {DATA_DIR}.")
    print("Eseguire prima semantic_search_baseline.ipynb per generarlo.")
    print("\nMetriche BM25:")
    print(json.dumps(metrics["overall"], indent=2))

## 9. Salvataggio output

In [ ]:
write_jsonl_atomic(results, RESULTS_OUTPUT_PATH)
write_json_atomic(metrics, METRICS_OUTPUT_PATH)

print(f"Salvati risultati: {RESULTS_OUTPUT_PATH}")
print(f"Salvate metriche: {METRICS_OUTPUT_PATH}")